# 6. Network Optimization via Genetic Algorithm

## Overview
This notebook implements a **Genetic Algorithm (GA)** to optimize the placement of new EV charging stations. The GA searches for optimal locations that maximize a **multi-objective fitness function** balancing:

| Metric | Weight | Objective |
|--------|--------|-----------|
| Average Distance | 30% | Minimize (stations closer together) |
| Coverage Area Ratio | 40% | Maximize (better geographic coverage) |
| Diameter | 10% | Minimize (reduce worst-case path length) |
| Clustering | 10% | Optimize connectivity patterns |
| Density | 10% | Maximize interconnectedness |

### GA Components
- **Initial Population**: Generated using pre-trained Random Forest models (Notebook 05) to produce geographically plausible candidate locations
- **Selection**: Tournament selection (size 2)
- **Crossover**: Single-point crossover with geographic validity checks
- **Mutation**: Random perturbation (±1°) with Germany boundary constraints
- **Elitism**: Top 5% of solutions preserved across generations
- **Stopping Criterion**: No improvement for 10 consecutive generations

In [ ]:
import networkx as nx
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import os
import sys
import random
import pickle
import joblib
from shapely.geometry import Point, MultiPoint
from sklearn import preprocessing
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')

sys.path.insert(0, '../src')
from utils import get_osrm_distance, draw_graph, calculate_weighted_metrics, weighted_mean

## 6.1 Load Data and Geographic Boundaries

In [ ]:
# Load charging station data
data = pd.read_csv('../data/processed/ChargingStationCleaned.csv', encoding='utf-8')
state_corrections = {
    'Baden-Wï¿½rttemberg': 'Baden-Württemberg',
    'Thï¿½ringen': 'Thüringen',
}
data['federal_state'] = data['federal_state'].replace(state_corrections)
data['geometry'] = data.apply(
    lambda row: Point(row['longitude_[dg]'], row['latitude_[dg]']), axis=1
)

# Load Germany boundary polygon for geographic constraint enforcement
germany_boundary = gpd.read_file('../data/boundaries/4_niedrig.geo.json')
germany_polygon = germany_boundary.geometry.unary_union

# Load pre-computed network graphs
years = [2010, 2011, 2012, 2013]
graphs = {}
for year in years:
    file_name = f'../results/networks/baseline/network_{year}_100.graphml'
    print(f'Loading network for {year}...')
    graphs[year] = nx.read_graphml(file_name)

## 6.2 Genetic Algorithm Implementation

### Architecture
```
┌─────────────────────────┐
│  Initial Population     │  ← RF model predictions + road snapping
│  (N candidate configs)  │
└──────────┬──────────────┘
           │
           ▼
┌─────────────────────────┐
│  Fitness Evaluation     │  ← Multi-objective: distance, coverage,
│  (5 weighted metrics)   │    diameter, clustering, density
└──────────┬──────────────┘
           │
           ▼
┌─────────────────────────┐
│  Tournament Selection   │  ← Select parents (tournament size = 2)
└──────────┬──────────────┘
           │
           ▼
┌─────────────────────────┐
│  Crossover + Mutation   │  ← Geographic validity enforced
└──────────┬──────────────┘
           │
           ▼
┌─────────────────────────┐
│  Elitism (top 5%)       │  ← Preserve best solutions
└──────────┬──────────────┘
           │
     Repeat until convergence
```

In [ ]:
def genetic_algorithm(graph, population_size=100, num_generations=100,
                      mutation_rate=0.1, num_new_stations=50, year=2022):
    """
    Optimize EV charging station placement using a Genetic Algorithm.
    
    Args:
        graph: Existing network graph (NetworkX) to be augmented.
        population_size: Number of candidate solutions per generation.
        num_generations: Maximum number of GA iterations.
        mutation_rate: Initial probability of mutating each station location.
        num_new_stations: Number of new stations to add to the network.
        year: Target year for the prediction model.
    
    Returns:
        Optimized NetworkX graph with new stations added at optimal locations.
    """
    initial_mutation_rate = 0.8
    tournament_size = 2
    no_improvement_threshold = 10  # Early stopping: generations without improvement

    # ------------------------------------------------------------------
    # INITIAL POPULATION GENERATION
    # Uses pre-trained RF models to predict plausible station locations,
    # distributed proportionally across federal states.
    # ------------------------------------------------------------------
    def generate_initial_population(graph, population_size, num_new_stations,
                                    germany_polygon, year):
        """Generate candidate station configurations using RF predictions."""
        filename = f'../results/networks/population_{year}.pkl'

        # Load cached population if available (expensive to compute)
        if os.path.exists(filename):
            with open(filename, 'rb') as f:
                population = pickle.load(f)
            print(f'Loaded cached population for {year}')
            return population

        # Load the year-specific Random Forest models
        rf_model_lat = joblib.load(f'../models/{year}/rf_model_{year}_lat.pkl')
        rf_model_lon = joblib.load(f'../models/{year}/rf_model_{year}_lon.pkl')

        # Encode federal states for model input
        le = preprocessing.LabelEncoder()
        federal_states = data['federal_state'].unique()
        le.fit(federal_states)

        # Distribute new stations proportionally to existing state distribution
        state_counts = data['federal_state'].value_counts()
        state_proportions = state_counts / state_counts.sum()
        num_stations_per_state = {
            state: int(round(num_new_stations * prop))
            for state, prop in state_proportions.items()
        }

        # Ensure total matches target
        total_new = sum(num_stations_per_state.values())
        if total_new < num_new_stations:
            max_state = max(num_stations_per_state, key=num_stations_per_state.get)
            num_stations_per_state[max_state] += num_new_stations - total_new

        population = []
        for _ in tqdm(range(population_size), desc='Creating initial population'):
            new_stations = []
            for state, num_stations in num_stations_per_state.items():
                state_encoded = le.transform([state])[0]
                for _ in range(num_stations):
                    # Predict coordinates using RF model
                    pred_lat = rf_model_lat.predict(
                        pd.DataFrame([[year, state_encoded]],
                                     columns=['year', 'federal_state_encoded'])
                    )
                    pred_lon = rf_model_lon.predict(
                        pd.DataFrame([[year, state_encoded]],
                                     columns=['year', 'federal_state_encoded'])
                    )

                    # Ensure predicted point is within Germany's borders
                    new_point = Point(pred_lon[0], pred_lat[0])
                    while not germany_polygon.contains(new_point):
                        pred_lat = rf_model_lat.predict([[year, state_encoded]])
                        pred_lon = rf_model_lon.predict([[year, state_encoded]])
                        new_point = Point(pred_lon[0], pred_lat[0])

                    new_stations.append((pred_lat[0], pred_lon[0]))
            population.append(new_stations)

        # Cache the generated population for future runs
        os.makedirs(os.path.dirname(filename), exist_ok=True)
        with open(filename, 'wb') as f:
            pickle.dump(population, f)

        return population

    # ------------------------------------------------------------------
    # NODE INSERTION
    # Adds a new station to the graph and connects it to nearby stations.
    # ------------------------------------------------------------------
    def add_node_to_graph(graph, new_node_coordinates, max_distance):
        """Add a new charging station node and connect to nearby stations."""
        new_node_id = len(graph) + 1
        lat, lon = new_node_coordinates
        graph.add_node(new_node_id, latitude=lat, longitude=lon)

        # Connect to all existing nodes within the distance threshold
        for node_id in graph.nodes:
            if node_id != new_node_id:
                lat2 = graph.nodes[node_id]['latitude']
                lon2 = graph.nodes[node_id]['longitude']
                distance = get_osrm_distance(lat, lon, lat2, lon2)
                if distance is not None and distance < max_distance * 1000:
                    graph.add_edge(new_node_id, node_id, weight=distance)
        return graph

    # ------------------------------------------------------------------
    # COVERAGE METRIC
    # Computes the ratio of the network's convex hull area to Germany's area.
    # ------------------------------------------------------------------
    def calculate_network_to_country_area_ratio(graph, country_polygon):
        """Compute network coverage as convex hull area / country area."""
        node_coords = [
            (graph.nodes[n]['longitude'], graph.nodes[n]['latitude'])
            for n in graph.nodes()
        ]
        network_polygon = MultiPoint(node_coords).convex_hull
        return network_polygon.area / country_polygon.area

    # ------------------------------------------------------------------
    # FITNESS FUNCTION
    # Multi-objective weighted evaluation of a candidate solution.
    # Lower fitness = better solution (minimization problem).
    # ------------------------------------------------------------------
    def fitness_function(solution, graph, max_distance=100):
        """Evaluate a candidate solution using weighted network metrics."""
        updated_graph = graph.copy()
        for coords in solution:
            updated_graph = add_node_to_graph(updated_graph, coords, max_distance)

        metrics = calculate_weighted_metrics(updated_graph, year)
        coverage = calculate_network_to_country_area_ratio(updated_graph, germany_polygon)

        # Min-max normalization using observed bounds from historical data
        norm_avg_dist = (metrics['average_distance'] - 15.10) / (58.43 - 15.20)
        norm_diameter = (metrics['diameter'] - 1) / (11 - 1)
        norm_clustering = (metrics['average_clustering'] - 0.72) / (0.93 - 0.72)
        norm_density = (metrics['density'] - 0.08) / (1 - 0.08)

        # Weighted combination (lower = better)
        # Weight allocation: 30% distance, 10% diameter, 10% clustering,
        #                    10% density, 40% coverage
        weights = [0.3, 0.1, 0.1, 0.1, 0.4]
        values = [
            norm_avg_dist,
            norm_diameter,
            norm_clustering,
            1 - norm_density,     # Invert: higher density is better
            1 - coverage          # Invert: higher coverage is better
        ]
        return weighted_mean(values, weights)

    # ------------------------------------------------------------------
    # GENETIC OPERATORS
    # ------------------------------------------------------------------
    def crossover(parent1, parent2):
        """Single-point crossover with geographic validity enforcement."""
        point = random.randint(1, len(parent1) - 1)
        child1 = parent1[:point] + parent2[point:]
        child2 = parent2[:point] + parent1[point:]

        # Revert invalid points (outside Germany) to parent values
        for i, (lat, lon) in enumerate(child1):
            if not germany_polygon.contains(Point(lon, lat)):
                child1[i] = parent1[i]
        for i, (lat, lon) in enumerate(child2):
            if not germany_polygon.contains(Point(lon, lat)):
                child2[i] = parent2[i]
        return child1, child2

    def mutate(solution, mutation_rate, germany_polygon):
        """Randomly perturb station locations (±1°) within Germany."""
        for i in range(len(solution)):
            if random.random() < mutation_rate:
                lat, lon = solution[i]
                new_point = Point(
                    lon + random.uniform(-1, 1),
                    lat + random.uniform(-1, 1)
                )
                # Resample until the mutated point is within Germany
                while not germany_polygon.contains(new_point):
                    new_point = Point(
                        lon + random.uniform(-1, 1),
                        lat + random.uniform(-1, 1)
                    )
                solution[i] = (new_point.y, new_point.x)
        return solution

    def tournament_selection(population, fitness_values, tournament_size):
        """Select the best individual from a random tournament."""
        indices = random.sample(range(len(population)), tournament_size)
        best = min(indices, key=lambda i: fitness_values[i])
        return population[best]

    # ------------------------------------------------------------------
    # MAIN GA LOOP
    # ------------------------------------------------------------------
    population = generate_initial_population(
        graph, population_size, num_new_stations, germany_polygon, year
    )

    best_solution = None
    best_fitness = float('-inf')
    no_improvement_count = 0

    for generation in range(num_generations):
        # Evaluate fitness for all solutions
        fitness_values = [fitness_function(s, graph) for s in population]

        # Track best solution
        current_best = min(fitness_values)
        if best_fitness == float('-inf') or current_best < best_fitness:
            best_fitness = current_best
            best_solution = population[fitness_values.index(best_fitness)]
            no_improvement_count = 0
        else:
            no_improvement_count += 1

        print(f'Generation {generation:3d} | Best Fitness: {best_fitness:.6f}')

        # Early stopping check
        if no_improvement_count >= no_improvement_threshold:
            print(f'Converged (no improvement for {no_improvement_threshold} generations)')
            break

        # Elitism: preserve top solutions
        num_elites = max(1, int(0.05 * population_size))

        # Tournament selection for the rest
        new_pop = [
            tournament_selection(population, fitness_values, tournament_size)
            for _ in range(population_size - num_elites)
        ]

        # Crossover and mutation
        offspring = []
        if len(new_pop) % 2 != 0:
            new_pop.append(new_pop[-1])

        for i in range(0, len(new_pop), 2):
            c1, c2 = crossover(new_pop[i], new_pop[i + 1])
            c1 = mutate(c1, initial_mutation_rate, germany_polygon)
            c2 = mutate(c2, initial_mutation_rate, germany_polygon)
            offspring.extend([c1, c2])

        # Preserve elite solutions
        elite_idx = sorted(
            range(len(fitness_values)),
            key=lambda i: fitness_values[i]
        )[:num_elites]
        elites = [population[i] for i in elite_idx]
        population = elites + offspring

    # Build the optimized graph with the best solution
    best_solution = min(population, key=lambda x: fitness_function(x, graph))
    optimized_graph = graph.copy()
    for coords in best_solution:
        optimized_graph = add_node_to_graph(optimized_graph, coords, max_distance=100)

    return optimized_graph

## 6.3 Run Optimization

Apply the genetic algorithm to selected years. The number of new stations to add is determined by the difference between consecutive years in the real data.

In [ ]:
# Example: Optimize the 2011 network by adding 174 new stations
year = 2011
num_new = 174

optimized_network = genetic_algorithm(
    graphs[year],
    num_new_stations=num_new,
    num_generations=30,
    year=year,
    population_size=60,
    mutation_rate=0.8
)

# Save the optimized network
os.makedirs('../results/networks/optimized', exist_ok=True)
nx.write_graphml(optimized_network, f'../results/networks/optimized/network_{year+1}_optimized.graphml')
print(f'Optimized network saved: {optimized_network.number_of_nodes()} nodes, {optimized_network.number_of_edges()} edges')

## 6.4 Visualize Optimized Network

In [ ]:
# Visualize the optimized network overlaid on Germany
draw_graph(optimized_network, title=f'Optimized EV Charging Network ({year+1})')